# Understanding CSLS, and the three pruning rules

In [6]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from corpora import load
from schemaoracle import cache, pruning

S = ["age", "gender", "sex", "tumor_size", "patient_notes"]
T = ["age_at_diagnosis", "gender", "tumor_size_mm", "ethnicity", "generic_id"]
M = np.array([
    [0.82, 0.30, 0.25, 0.22, 0.55],
    [0.28, 0.85, 0.22, 0.24, 0.55],
    [0.26, 0.70, 0.20, 0.22, 0.55],
    [0.24, 0.20, 0.62, 0.18, 0.72],
    [0.22, 0.22, 0.22, 0.22, 0.34]])

pd.DataFrame(M, S, T)

,age_at_diagnosis,gender,tumor_size_mm,ethnicity,generic_id
age,0.82,0.30,0.25,0.22,0.55
gender,0.28,0.85,0.22,0.24,0.55
sex,0.26,0.70,0.20,0.22,0.55
tumor_size,0.24,0.20,0.62,0.18,0.72
patient_notes,0.22,0.22,0.22,0.22,0.34


## The problem CSLS solves


    csls(s, t) = 2 cos(s, t) - mean cos(s, s's k nearest targets)
                             - mean cos(t, t's k nearest sources)


In [7]:
K = pruning.CSLS_K

r_src = np.sort(M, axis=1)[:, -K:].mean(axis=1)   # each source's own neighbourhood
r_tgt = np.sort(M, axis=0)[-K:, :].mean(axis=0)   # each target's own
C = 2 * M - r_src[:, None] - r_tgt[None, :]

assert np.allclose(C, pruning.csls(M)), "not the shipped correction"

print("tumor_size, before and after:")
display(pd.DataFrame({"cosine": M[3], "csls": C[3]}, T)
        .sort_values("cosine", ascending=False).round(3))
print(f"raw cosine picks  {T[M[3].argmax()]}")
print(f"csls picks        {T[C[3].argmax()]}")

tumor_size, before and after:


,cosine,csls
generic_id,0.72,0.506
tumor_size_mm,0.62,0.546
age_at_diagnosis,0.24,-0.276
gender,0.20,-0.446
ethnicity,0.18,-0.248


raw cosine picks  generic_id
csls picks        tumor_size_mm


## The three rules

Each is one comparison on that corrected matrix.

| rule | question | drops |
|---|---|---|
| `mutual` | does my best target rank me first in return? | a source column |
| `zerosrc` | is any target closer to me than my own neighbourhood? | a source column |
| `zero` | is any source closer to me than my own neighbourhood? | a target column |


In [10]:
pruning.mutual_depth(C)

array([1, 1, 2, 1, 1])

In [12]:
K

10

In [8]:
mutual = pruning.mutual_depth(C) > 1    
zerosrc = C.max(axis=1) <= 0            
zero = C.max(axis=0) <= 0               

assert (pruning.masks(M, ("mutual",))[0] == mutual).all()
assert (pruning.masks(M, ("zerosrc",))[0] == zerosrc).all()
assert (pruning.masks(M, ("zero",))[1] == zero).all()

display(pd.DataFrame({"best target": [T[j] for j in C.argmax(axis=1)],
                      "best csls": C.max(axis=1).round(3),
                      "mutual drops": mutual,
                      "zerosrc drops": zerosrc}, S))
display(pd.DataFrame({"best source": [S[i] for i in C.argmax(axis=0)],
                      "best csls": C.max(axis=0).round(3),
                      "zero drops": zero}, T))

,best target,best csls,mutual drops,zerosrc drops
age,age_at_diagnosis,0.848,False,False
gender,gender,0.818,False,False
sex,gender,0.560,True,False
tumor_size,tumor_size_mm,0.546,False,False
patient_notes,ethnicity,-0.020,False,True


,best source,best csls,zero drops
age_at_diagnosis,age,0.848,False
gender,gender,0.818,False
tumor_size_mm,tumor_size,0.546,False
ethnicity,patient_notes,-0.020,True
generic_id,tumor_size,0.506,False


Read the two tables together and the rules separate cleanly.

`sex` is dropped by `mutual` alone. It has a real best target, `gender`, and
that target prefers the column actually called `gender`. Asking about `sex` is
asking about a target already spoken for.

`patient_notes` is dropped by `zerosrc` alone. Nothing in the schema is closer
to it than its own neighbourhood, so there is no candidate worth a call. Note
that `mutual` keeps it: its best target is `ethnicity`, and `ethnicity`'s best
source is `patient_notes`, so the two are mutual nearest neighbours. They are
a mutually unwanted pair, and `mutual` cannot tell that apart from a real
match. That is the gap `zerosrc` closes.

`ethnicity` is dropped by `zero`: no source column clears it, so it is dead
weight in every prompt it appears in.

## On a real study

Same four lines, read off a cached retrieval order. Nothing is sent.

In [9]:
PAIR = "Gillette"
p = next(x for x in load("gdc") if x["name"] == PAIR)
raw = json.loads(cache.retrieval_path(REPO / "results/runs", "gdc", PAIR, "qwen")
                 .read_text(encoding="utf-8"))["rankings"]
Mr, Sr, Tr = pruning.score_matrix({k: [(t, float(v)) for t, v in r]
                                   for k, r in raw.items()})
Cr = pruning.csls(Mr)
gold = {str(s) for s, _ in p["ground_truth"]}

D = pd.DataFrame({"has gold": [s in gold for s in Sr],
                  "best target": [Tr[j] for j in Cr.argmax(axis=1)],
                  "best csls": Cr.max(axis=1).round(3),
                  "mutual": pruning.mutual_depth(Cr) > 1,
                  "zerosrc": Cr.max(axis=1) <= 0}, Sr)

print(f"{PAIR}: {len(Sr)} source columns, {len(Tr)} targets, "
      f"{len(gold)} with a gold target")
print("the 12 weakest columns, where the cut happens:")
display(D.nsmallest(12, "best csls"))
for rule in ("mutual", "zerosrc"):
    cut = D[D[rule]]
    print(f"{rule:8s} drops {len(cut):3d} of {len(D)}, "
          f"{int(cut['has gold'].sum())} of them with gold")

Gillette: 67 source columns, 736 targets, 15 with a gold target
the 12 weakest columns, where the cut happens:


,has gold,best target,best csls,mutual,zerosrc
KEAP1.mutation.status,False,papillary_renal_cell_type,-0.087,True,True
ERBB2.mutation,False,annotated_somatic_mutations,-0.059,True,True
KEAP1.mutation,False,molecular_consequence,-0.049,True,True
ERBB2.mutation.status,False,enneking_msts_metastasis,-0.020,True,True
Dominant.Signature.Fraction.WGS.notSmoking,False,no_matched_normal_wgs,-0.019,True,True
Smoking.Signature.Fraction.WGS,False,years_smoked,-0.016,True,True
Dominant.Signature.WGS.notSmoking.50perc,False,somatic_annotation_workflows,-0.015,True,True
STK11.mutation,False,simple_somatic_mutations,-0.004,True,True
NMF.cluster.membership,False,morphology,-0.001,False,True
Mutation.Signature.Activity.W2.COSMIC4,False,mutation_codon,0.001,True,False


mutual   drops  30 of 67, 1 of them with gold
zerosrc  drops   9 of 67, 0 of them with gold


The cut is visible in that table. Read down `best csls` and the sign
changes between `NMF.cluster.membership` at -0.001 and
`Mutation.Signature.Activity.W2.COSMIC4` at +0.001, and `zerosrc` changes with
it. That is the whole rule, and it is why it needs no threshold to tune: zero
is where "closer than my own neighbourhood" stops being true.

`NMF.cluster.membership` is also the real version of `patient_notes`. It is
dropped by `zerosrc` and kept by `mutual`, because it and `morphology` are each
other's nearest neighbour while neither is close to anything.

The column to watch is `has gold` against the drop flags. A dropped column
with no gold is free: on `rerank`, which cannot decline, it was going to be a
false positive whatever the model said. A dropped column with gold is recall
paid out for that precision.

Pooled over all ten GDC studies, `mutual` removes 234 of 569 columns and 12 of
those have gold; `zerosrc` removes 81 and none of them do.